In [ ]:
import pyscf

import numpy as np

import matplotlib.pyplot as plt

In [ ]:
# Updated MD step excluding frozen atoms
def velocity_verlet(atom_coords, velocities, forces, dt, mass, frozen_mask):
    # Only update non-frozen atoms
    unfrozen_mask = ~frozen_mask  # Invert the mask
    
    # Update positions only for unfrozen atoms
    atom_coords[unfrozen_mask] += velocities[unfrozen_mask] * dt + 0.5 * forces[unfrozen_mask] / mass * dt**2
    
    # Compute new forces after position update
    new_forces = compute_forces(atom_coords)  # This uses PySCF to compute the forces
    
    # Update velocities only for unfrozen atoms
    velocities[unfrozen_mask] += 0.5 * (forces[unfrozen_mask] + new_forces[unfrozen_mask]) / mass * dt
    
    return atom_coords, velocities, new_forces


In [ ]:
from pyscf.pbc.tools import pbc as pbctools

# Central atom (assume atom 0, Na, for example)
central_atom_index = 0

# Function to calculate distances
def calculate_distances(atom_coords, central_index):
    distances = np.linalg.norm(atom_coords - atom_coords[central_index], axis=1)
    return distances

# Get atom coordinates
atom_coords = cell.atom_coords()
distances = calculate_distances(atom_coords, central_atom_index)

# Function to freeze atoms within a certain radius
def freeze_atoms_within_radius(mf, distances, radius):
    # Create a mask of frozen atoms
    frozen_mask = distances < radius
    # Apply frozen mask to the model (pseudo-code, you'll need actual PySCF freezing method)
    # Freeze atoms using the mask (set constraints in the force calculation)
    return frozen_mask


In [ ]:
# Freeze atoms and generate perturbations for each radius
radii = np.linspace(1.0, 10.0, 10)  # Example radii in Angstroms
num_perturbations = 50  # Number of MD or MC steps for each radius
std_forces = []

for radius in radii:
    frozen_mask = freeze_atoms_within_radius(mf, distances, radius)
    
    # Initialize atom positions, velocities (for MD), and forces
    atom_coords = cell.atom_coords()
    velocities = np.zeros_like(atom_coords)  # Start with zero velocities for MD
    forces = compute_forces(atom_coords)  # Use PySCF to compute forces
    
    forces_list = []
    
    # Generate perturbations using MD or MC, excluding frozen atoms
    for _ in range(num_perturbations):
        # Use MD or MC to update atom positions and velocities
        atom_coords, velocities, forces = velocity_verlet(atom_coords, velocities, forces, dt, mass, frozen_mask)
        
        # Store force on the central atom after each perturbation
        central_force = forces[central_atom_index]
        forces_list.append(central_force)
    
    # Compute the standard deviation of forces on the central atom
    forces_array = np.array(forces_list)
    std_forces.append(np.std(forces_array, axis=0))

# Plot the standard deviation of forces as a function of the frozen radius
plt.plot(radii, std_forces)
plt.xlabel("Radius of frozen sphere (Angstroms)")
plt.ylabel("Standard deviation of force on central atom (a.u.)")
plt.title("Locality Test: Force Std Dev vs Frozen Radius")
plt.show()
